In [ ]:
%load_ext autoreload
%autoreload 2
%run ./initialize.ipynb

In [ ]:
import lion_tools
import ipynbname
spark=lion_tools.get_or_create_spark()
spark.createDataFrame([[1, "Alice]\\n in hell{}"], [2, "Bob"]], ["id", "name"]).display(
    file_path=str(ipynbname.path().parent.parent.joinpath("output", "unknown_dataframe_view.html"))
)

In [ ]:
df = spark.read.parquet(str(ipynbname.path().parent.parent.joinpath("datasets", "movies.parquet"))).display(column_filter=True)

In [ ]:
import ipynbname
import pyspark.sql.functions as F
df = spark.read.parquet(str(ipynbname.path().parent.parent.joinpath("datasets", "movies.parquet")))
# df.eGroup('Distributor').count().eD()
df = (
    df
    .withColumn('piv', (F.floor(F.rand() * 3) + 1).cast('int'))
    .withColumn('piv', F.expr("if(piv = 1, null, piv)"))
)
# (
#     df
#     .group('Distributor') # type: ignore
#     .pivot('piv', 'column_grouped')
#     .agg('count(distinct Director)', 'avg(`IMDB Rating`)', 'sum(`IMDB Votes`)') # type: ignore
#     # .agg('count(distinct Director)') # type: ignore
#     # .sum()
#     # .display(p=5, compact=1)
#     # .show(5, truncate=False)
# )
(
    df
    .group('Distributor') # type: ignore
    .pivot('piv')
    .totals()
    .agg('count(distinct Director)', 'avg(`IMDB Rating`)', 'sum(`IMDB Votes`)') # type: ignore
    # .agg('count(distinct Director)') # type: ignore
    # .sum()
    .display(p=30, compact=1)
    # .show(5, truncate=False)
)
# (
#     df
#     .group('Distributor') # type: ignore
#     .pivot('piv', 'default')
#     .totals()
#     .agg('count(distinct Director)', 'avg(`IMDB Rating`)', 'sum(`IMDB Votes`)') # type: ignore
#     # .agg('count(distinct Director)') # type: ignore
#     # .sum()
#     # .display(p=3, compact=1)
#     # .show(5, truncate=False)
# )

# (df.display(p=1))


# df.eDisplay(sort=-4, page_length=1)
# df.selectExpr('*').display(p=1, file_path=str(ipynbname.path().parent.parent.joinpath("output", "movies_dataframe_view_1.html")))
# df.selectExpr('*').display(p=10, file_path=str(ipynbname.path().parent.parent.joinpath("output", "movies_dataframe_view_10.html")))
# df.selectExpr('*').display(p=20, file_path=str(ipynbname.path().parent.parent.joinpath("output", "movies_dataframe_view_20.html")))
# df.eDisplay(sort=-4, page_length=15)
# df.eDisplay(sort=-4, page_length=100)
# df.eDisplay(file_path=str(ipynbname.path().parent.parent.joinpath("output", "movies_dataframe_view.html")))
# df.select("Creative Type", "Director", "Distributor", "IMDB Rating", ).eDisplay()


In [ ]:

E.show_schema(df)

In [ ]:
from pyspark.sql.types import (
    ArrayType, IntegerType, MapType, StringType, StructField, StructType,
)

nested_schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("address", StructType([
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
        StructField("geo", StructType([
            StructField("lat", StringType(), True),
            StructField("lon", StringType(), True),
        ]), True),
    ]), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("orders", ArrayType(StructType([
        StructField("order_id", IntegerType(), False),
        StructField("items", ArrayType(StructType([
            StructField("sku", StringType(), True),
            StructField("qty", IntegerType(), True),
        ])), True),
    ])), True),
    StructField("attributes", MapType(StringType(), StringType()), True),
])

nested_df = spark.createDataFrame(
    [
        (
            1,
            "Alice",
            ("Main St", "Springfield", ("12.34", "56.78")),
            ["vip", "newsletter"],
            [(100, [("sku-1", 2), ("sku-2", 1)])],
            {"color": "blue", "size": "M"},
        ),
    ],
    nested_schema,
)

from lion_tools.DataFrameOther import DataFrameOther as dfo
from IPython.display import HTML
display(HTML(dfo.show_schema_html(nested_df)))

In [ ]:
lion_tools.DataFrameDisplay.display(
    df.selectExpr('struct(*) as xxxx', '*'),
    n=10,
    file_path=str(ipynbname.path().parent.parent.joinpath("output", "movies_dataframe_view.html"))
)

In [ ]:
df.selectExpr('*').display(n=10, file_path=str(ipynbname.path().parent.parent.joinpath("output", "movies_dataframe_view_good.html")))